# Multi-Head Attention（LLaMA 风格，含 KV Cache）

## 结构
- 四个投影：`q_proj / k_proj / v_proj / o_proj`，无 bias。
- 前向：投影 → reshape 到 `[B, h, N, d_k]` → scaled-dot-product → 拼接 → `o_proj`。
- **KV Cache**：自回归解码时每步只新增 1 token，把历史 K/V 缓存，新步只算新 token 的 q/k/v 并 `cat` 进缓存，避免每步重算全部历史（复杂度 $O(n^2)\to O(n)$）。
  - prefill：一次算 prompt 全部 K/V 填入 cache。
  - decode：每步 q 是 `[B,1,...]`，k/v cat 到 cache 后 attention 对全部历史做。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LlamaMHA(nn.Module):
    def __init__(self, dim, num_heads, head_dim=None):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim if head_dim is not None else dim // num_heads
        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x, attention_mask=None, cache=None):
        B, N, _ = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        new_cache = (k, v)
        attn = (q @ k.transpose(-1, -2)) * self.scale
        if attention_mask is not None:
            attn = attn + attention_mask
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, N, -1)
        return self.o_proj(out), new_cache

In [ ]:
# 验证：prefill + 逐 token decode，结果应与一次性算全长一致
torch.manual_seed(0)
dim, heads = 64, 8
mha = LlamaMHA(dim, heads)
x_full = torch.randn(1, 5, dim)

# 一次性
out_full, _ = mha(x_full)

# prefill 前 3 + decode 后 2
out_pre, cache = mha(x_full[:, :3])
outs = [out_pre]
for t in range(3, 5):
    o, cache = mha(x_full[:, t:t+1], cache=cache)
    outs.append(o)
out_inc = torch.cat(outs, dim=1)
print('增量与全长一致:', torch.allclose(out_inc, out_full, atol=1e-5))
print('shape:', out_inc.shape)

## 小结 / 易错点
- KV cache 的 `cat` 在 `dim=2`（序列维），注意 head 维在 `dim=1`。
- decode 阶段 q 长度为 1，但 k/v 是全部历史，attention 输出仍是 `[B,h,1,d_k]`。
- 训练时不用 cache；推理 prefill 后接 decode 是标准流程。
- `attention_mask` 加法形式（$-\infty$ 填充）比乘法形式更稳。